# Notebook 09: 强化学习交易智能体 — 环境走查 + PPO 训练

## 本节目标
1. 理解 Gymnasium RL 环境接口 (reset/step/render)
2. 使用 ElectricityMarketEnv 构建电力交易环境
3. 实现随机策略 (Random Policy) 作为基线对比
4. 训练 PPO (Proximal Policy Optimization) 智能体进行自动投标
5. 评估训练效果：训练曲线、考核回放、动作分布

## 前置条件
- 完成 Phase 1-2 notebooks (01-05)
- 安装 stable-baselines3 >= 2.8.0, gymnasium >= 1.3.0
- 推荐: 数据文件 `data/price_data.xlsx`（ZionLuo 电价数据集）

> **RL 背景**: PPO 是 OpenAI 2017 年提出的策略梯度算法，
> 通过 clipped surrogate objective 限制策略更新步长，
> 是目前最稳定、最常用的 on-policy RL 算法之一。

In [ ]:
# ── 导入依赖 ──────────────────────────────────
import sys
sys.path.insert(0, '../..')  # 让 Python 能找到 ellectric/ 包

import os
import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level=logging.WARNING)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

import gymnasium as gym
from stable_baselines3.common.monitor import Monitor

from ellectric.pipeline.data_loader import create_loader
from ellectric.pipeline.cleaner import clean_data
from ellectric.pipeline.price_loader import create_price_loader
from ellectric.pipeline.forecaster import XGBoostForecaster
from ellectric.pipeline.price_forecaster import LEARForecaster
from ellectric.pipeline.features import FeatureEngineer
from ellectric.pipeline.trading_env import ElectricityMarketEnv, RewardRegistry
from ellectric.pipeline.rl_trainer import RLAgentFactory

print("✅ 所有依赖导入成功")
print(f"  gymnasium: {gym.__version__}")
print(f"  pandas: {pd.__version__}")
print(f"  可用奖励函数: {RewardRegistry.list()}")

---
## 步骤 1: 加载数据

使用 `create_price_loader()` 加载 ZionLuo 电价数据集（含日前价格、统调负荷等）。
如果数据文件不存在，自动生成合成数据以供后续测试。

`PriceDataLoader` 返回 7 列标准化数据:
| 列名 | 含义 | 单位 |
|------|------|------|
| `timestamp` | UTC 时间戳 | datetime |
| `price_da` | 日前价格 | 元/MWh |
| `load_mw` | 统调负荷 | MW |
| `wind_mw` | 风电出力 | MW |
| `solar_mw` | 光伏出力 | MW |
| `tie_line_mw` | 省间联络线 | MW |

In [ ]:
# ── 尝试加载真实数据，失败则用合成数据 ──────────
DATA_PATH = '../../data/price_data.xlsx'

price_loader = create_price_loader(DATA_PATH)

try:
    df_price = price_loader.load_data()
    print(f"✅ 加载真实数据: {len(df_price)} 行")
    print(f"   时间范围: {df_price['timestamp'].min()} ~ {df_price['timestamp'].max()}")

    # 从价格数据中提取负荷数据
    df_load = df_price[['timestamp', 'load_mw']].copy()
    print(f"   负荷数据: {len(df_load)} 行")

except (FileNotFoundError, Exception) as e:
    print(f"⚠️ 未找到真实数据 ({e})")
    print("→ 生成合成小时级数据用于演示...")

    n_hours = 24 * 180  # 6 个月
    timestamps = pd.date_range(start='2024-01-01', periods=n_hours, freq='h')
    hour = timestamps.hour
    dow = timestamps.dayofweek

    rng = np.random.RandomState(42)
    hour_rad = 2 * np.pi * (hour - 6) / 24
    load_mw = 800 + 300 * np.sin(hour_rad) + 80 * np.sin(2 * np.pi * dow / 7)
    load_mw += 30 * rng.randn(n_hours)
    load_mw += np.linspace(0, 80, n_hours)

    price_da = 35 + 0.025 * load_mw + 8 * rng.randn(n_hours)
    price_da = np.maximum(price_da, 5)

    df_load = pd.DataFrame({'timestamp': timestamps, 'load_mw': load_mw})
    df_price = pd.DataFrame({
        'timestamp': timestamps,
        'price_da': price_da,
        'price_rt': price_da * 0.95,
        'load_mw': load_mw,
        'wind_mw': 100 + 30 * rng.randn(n_hours),
        'solar_mw': np.maximum(0, 200 * np.sin(np.pi * (hour - 6) / 12) + 20 * rng.randn(n_hours)),
        'tie_line_mw': 50 + 20 * rng.randn(n_hours),
    })
    print(f"✅ 合成数据已生成: {len(df_load)} 行")

print(f"\n负荷数据概况:\n{df_load['load_mw'].describe().to_string()}\n")
print(f"价格数据概况:\n{df_price['price_da'].describe().to_string()}")

---
## 步骤 2: 特征工程 + 训练预测器

ElectricityMarketEnv 需要负荷预测器和电价预测器来生成观测。
我们先训练一个简单的 XGBoost 负荷预测和 LEAR 电价预测。

> 完整的预测器训练见 Notebook 02-04。此处为快速训练（~30 秒）。
> 如果想跳过此步，环境会自动回退到持续法预测 (Persistence Forecast)。

In [ ]:
# ── 训练 XGBoost 负荷预测器 ────────────────────
print("=" * 50)
print("训练 XGBoost 负荷预测器...")
print("=" * 50)

engineer = FeatureEngineer()
df_feat_load = engineer.add_tier1_features(df_load)
feature_cols = engineer.get_feature_columns('tier1')

# 跳过 lag 引起的 NaN
train_start = 24
X_xgb = df_feat_load[feature_cols].iloc[train_start:]
y_xgb = df_feat_load['load_mw'].iloc[train_start:]

lf = XGBoostForecaster(n_estimators=50, max_depth=4)
xgb_result = lf.train_evaluate(X_xgb, y_xgb, n_splits=3, gap=24)
print(f"  负荷预测 MAE: {xgb_result['metrics']['mae']:.0f} MW")

# ── 训练 LEAR 电价预测器 ────────────────────────
print()
print("=" * 50)
print("训练 LEAR 电价预测器...")
print("=" * 50)

# 需要 load_mw 列用于 tier2 特征
df_feat_price = df_price.copy()
pf = LEARForecaster(alpha=0.01)
df_feat_price = pf.add_price_features(df_feat_price, 'tier1')
lear_result = pf.train_evaluate(df_feat_price, 'tier1', n_splits=3, gap=24)
print(f"  电价预测 MAE: {lear_result['metrics']['mae']:.2f} 元/MWh")
print(f"  电价预测 RMSE: {lear_result['metrics']['rmse']:.2f} 元/MWh")

print(f"\n✅ 预测器训练完成: XGBoost(MAE={xgb_result['metrics']['mae']:.0f}), LEAR(MAE={lear_result['metrics']['mae']:.2f})")

---
## 步骤 3: 初始化 ElectricityMarketEnv

`ElectricityMarketEnv` 是一个 Gymnasium 环境，模拟电力交易员的 24 小时投标决策。

### 观测空间 (Dict)
- `load_forecast_24h`: 未来 24 小时负荷预测 (24,)
- `price_forecast_24h`: 未来 24 小时电价预测 (24,)
- `time_features`: sin/cos 编码的时间特征 (4,)
- `price_history_168h`: 过去 168 小时电价历史 (168,)
- `account_state`: 账户状态 [cash, progress] (2,)

### 动作空间 (Box)
- Box(0, 1, (24,)) — 每小时的归一化投标量
- 实际投标量 (MW) = action × max_capacity

### 出清逻辑
- cleared = min(bid_mw, actual_load)  # 价格接受者
- P&L = -(bid_mw - actual_load) × price / 1000

In [ ]:
# ── 初始化 ElectricityMarketEnv ────────────────
# 使用训练好的预测器和数据
env = ElectricityMarketEnv(
    load_data=df_load,
    price_data=df_price,
    load_forecaster=lf,
    price_forecaster=pf,
    initial_cash=0.0,
    max_capacity=1000.0,
    reward_fn='profit_only',
)

print(f"观测空间: {env.observation_space}")
print(f"动作空间: {env.action_space}")
print(f"动作空间范围: [{env.action_space.low[0]}, {env.action_space.high[0]}]")
print(f"观测空间 keys: {list(env.observation_space.spaces.keys())}")
print()

# ── 测试 reset + step ───────────────────────────
obs, info = env.reset()
print(f"Reset 后观测 shapes:")
for k, v in obs.items():
    print(f"  {k}: {v.shape}")

action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)
print(f"\nStep 结果:")
print(f"  reward: {reward:.2f}")
print(f"  terminated: {terminated}")
print(f"  info keys: {list(info.keys())}")

---
## 步骤 4: 随机策略走查 (Random Policy Baseline)

在训练 RL 智能体之前，先用**随机策略**运行 100 个 episode，
建立基线性能。如果 PPO 的 reward 不能远超随机策略，说明有问题。

这也是理解环境的好方法：观察不同随机种子下的奖励分布。

In [ ]:
# ── 随机策略 100 episodes ───────────────────────
N_RANDOM_EPISODES = 100
random_rewards = []

for ep in range(N_RANDOM_EPISODES):
    obs, info = env.reset()
    done = False
    total_reward = 0.0

    while not done:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        done = terminated or truncated

    random_rewards.append(total_reward)

    if (ep + 1) % 25 == 0:
        print(f"  随机策略 Episode {ep + 1}/{N_RANDOM_EPISODES} ...")

random_rewards = np.array(random_rewards)

print(f"\n{'=' * 50}")
print(f"随机策略评估 ({N_RANDOM_EPISODES} episodes):")
print(f"{'=' * 50}")
print(f"  Mean reward: {random_rewards.mean():.2f}")
print(f"  Std reward:  {random_rewards.std():.2f}")
print(f"  Min reward:  {random_rewards.min():.2f}")
print(f"  Max reward:  {random_rewards.max():.2f}")
print(f"  Median:      {np.median(random_rewards):.2f}")

In [ ]:
# ── 随机策略奖励分布直方图 ──────────────────────
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=random_rewards,
    nbinsx=30,
    marker=dict(color='#1f77b4', line=dict(color='white', width=1)),
    name='随机策略',
))

fig.add_vline(
    x=random_rewards.mean(),
    line_dash='dash',
    line_color='red',
    annotation_text=f"均值 {random_rewards.mean():.2f}",
)

fig.update_layout(
    title=dict(text='随机策略 Reward 分布 (100 episodes)', font=dict(size=16)),
    xaxis_title='Episode Reward',
    yaxis_title='频次',
    height=450,
    hovermode='x',
    showlegend=False,
)
fig.show()

### 📖 观察随机策略

- 随机策略的 reward 均值约在 0 附近（有正有负）
- 标准差反映了环境的不确定性
  
> **思考**: 为什么随机策略有时也能获得正 reward？
> → 因为当 action 恰好接近真实负荷时，P&L 损失很小。
> → 随机策略大概 30% 左右的预测准确率。

---
## 步骤 5: PPO 训练

使用 stable-baselines3 的 PPO 算法训练交易智能体。
训练参数:
- 算法: PPO (MlpPolicy)
- 总步数: 50,000 timesteps
- TensorBoard 日志: `../../tb_logs/`
- 学习率: 0.0003 (默认)

> 💡 训练时间: CPU 上约 3-10 分钟 (取决于数据长度)。
> 如果想加速，可减少 `total_timesteps` 至 10000 先看效果。

In [ ]:
# ── PPO 训练 ────────────────────────────────────
TOTAL_TIMESTEPS = 50000  # 可修改: 先用 10000 快速测试

# 创建 Monitor 包装环境以记录 episode 奖励
train_env = Monitor(env, filename='../../tb_logs/monitor.csv')

# 创建 PPO 智能体
agent = RLAgentFactory.create(
    'ppo',
    train_env,
    tensorboard_log='../../tb_logs',
    verbose=1,
    learning_rate=0.0003,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
)

print(f"开始 PPO 训练 ({TOTAL_TIMESTEPS} timesteps)...")
print(f"TensorBoard 日志: ../../tb_logs/")
print(f"在终端运行: tensorboard --logdir ../../tb_logs")
print()

result = agent.train(total_timesteps=TOTAL_TIMESTEPS)

print(f"\n{'=' * 50}")
print(f"PPO 训练完成!")
print(f"{'=' * 50}")
print(f"  总步数: {result['total_timesteps']}")
print(f"  Final reward: {result['final_reward']:.2f}")
print(f"  算法: {result['algo'].upper()}")

# 保存模型
os.makedirs('../../models', exist_ok=True)
agent.save('../../models/ppo_trader.zip')
print(f"\n✅ 模型已保存到 ../../models/ppo_trader.zip")

---
## 步骤 6: 训练曲线

从 Monitor 日志中提取每个 episode 的奖励，观察 PPO 是否在学习。

In [ ]:
# ── 提取训练曲线 ────────────────────────────────

episode_rewards = train_env.get_episode_rewards()
episode_lengths = train_env.get_episode_lengths()

print(f"训练期间完成 {len(episode_rewards)} 个 episode")

# 计算滑动平均
window = max(1, len(episode_rewards) // 20)
if len(episode_rewards) > window:
    smoothed = pd.Series(episode_rewards).rolling(window=window, min_periods=1).mean()
else:
    smoothed = pd.Series(episode_rewards)

episode_indices = list(range(len(episode_rewards)))

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=('Episode Reward (PPO)', 'Episode Length'),
    row_heights=[0.6, 0.4],
)

fig.add_trace(
    go.Scatter(
        x=episode_indices, y=episode_rewards,
        mode='markers',
        name='Episode Reward',
        marker=dict(color='#1f77b4', size=3, opacity=0.5),
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=episode_indices, y=smoothed,
        mode='lines',
        name=f'滑动平均 (w={window})',
        line=dict(color='#d62728', width=2),
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(
        x=episode_indices, y=episode_lengths,
        mode='markers',
        name='Episode Length',
        marker=dict(color='#2ca02c', size=3, opacity=0.5),
    ),
    row=2, col=1,
)

fig.update_layout(
    height=600,
    hovermode='x unified',
    showlegend=True,
)
fig.update_xaxes(title_text='Episode', row=2, col=1)
fig.update_yaxes(title_text='Reward', row=1, col=1)
fig.update_yaxes(title_text='Steps', row=2, col=1)

# 添加基线
mean_random = float(random_rewards.mean())
fig.add_hline(
    y=mean_random, line_dash='dot', line_color='gray',
    annotation_text=f'随机策略均值 {mean_random:.2f}',
    row=1, col=1,
)

fig.show()

if episode_rewards:
    last_10 = np.mean(episode_rewards[-10:])
    print(f"最后 10 个 episode 平均 reward: {last_10:.2f}")
    print(f"随机策略平均 reward:          {mean_random:.2f}")
    print(f"Improvement: {last_10 - mean_random:+.2f}")

---
## 步骤 7: 评估训练好的智能体

运行 5 个完整 episode，记录每次交易的:
- 实际负荷 vs 智能体投标量
- 累计 P&L
- 出清价格

In [ ]:
# ── 创建独立的评估环境 ─────────────────────────
eval_env = ElectricityMarketEnv(
    load_data=df_load,
    price_data=df_price,
    load_forecaster=lf,
    price_forecaster=pf,
    initial_cash=0.0,
    max_capacity=1000.0,
    reward_fn='profit_only',
)

N_EVAL_EPISODES = 5
eval_results = []

for ep in range(N_EVAL_EPISODES):
    obs, info = eval_env.reset()
    done = False
    episode_data = {'actuals': [], 'bids': [], 'rewards': [], 'prices': []}

    while not done:
        action = agent.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        episode_data['actuals'].extend(info.get('actual_load', [0]))
        episode_data['bids'].extend((info.get('bid_mw', [0]) / 1000.0).tolist())
        episode_data['rewards'].append(reward)
        episode_data['prices'].extend(info.get('clearing_price', [0]))
        done = terminated or truncated

    eval_results.append(episode_data)
    total_r = np.sum(episode_data['rewards'])
    print(f"  Episode {ep + 1}: total_reward={total_r:.2f}")

In [ ]:
# ── 实际 vs 投标量叠加图 (第一个 episode) ─────────
ep = 0
data = eval_results[ep]

hour_labels = list(range(len(data['actuals'])))

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=('实际负荷 vs 投标量 (Episode 1)', '逐小时 Reward', '出清价格'),
    row_heights=[0.45, 0.25, 0.3],
)

fig.add_trace(
    go.Scatter(
        x=hour_labels, y=data['actuals'],
        mode='lines',
        name='实际负荷',
        line=dict(color='#1f77b4', width=2),
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=hour_labels, y=data['bids'],
        mode='lines',
        name='PPO 投标',
        line=dict(color='#ff7f0e', width=2, dash='dash'),
    ),
    row=1, col=1,
)

cum_rewards = np.cumsum(data['rewards'])
fig.add_trace(
    go.Scatter(
        x=list(range(len(data['rewards']))), y=data['rewards'],
        mode='lines+markers',
        name='Reward',
        line=dict(color='#2ca02c', width=1.5),
        marker=dict(size=4),
    ),
    row=2, col=1,
)

fig.add_trace(
    go.Scatter(
        x=hour_labels, y=data['prices'],
        mode='lines',
        name='出清价格',
        line=dict(color='#9467bd', width=1.5),
    ),
    row=3, col=1,
)

fig.update_layout(
    height=700,
    hovermode='x unified',
    showlegend=True,
)
fig.update_xaxes(title_text='Hour', row=3, col=1)
fig.update_yaxes(title_text='负荷 / 投标', row=1, col=1)
fig.update_yaxes(title_text='Reward', row=2, col=1)
fig.update_yaxes(title_text='价格 (元/MWh)', row=3, col=1)

fig.show()

---
## 步骤 8: 动作分布分析

分析智能体在不同小时的投标偏好。
好的交易策略应该根据负荷水平调整投标量。

In [ ]:
# ── 收集所有评估 episode 的动作 ─────────────────
all_actions = []

eval_env2 = ElectricityMarketEnv(
    load_data=df_load,
    price_data=df_price,
    load_forecaster=lf,
    price_forecaster=pf,
    initial_cash=0.0,
    max_capacity=1000.0,
    reward_fn='profit_only',
)

for ep in range(10):
    obs, info = eval_env2.reset()
    done = False
    while not done:
        action = agent.predict(obs, deterministic=True)
        all_actions.extend(action.tolist())
        obs, reward, terminated, truncated, info = eval_env2.step(action)
        done = terminated or truncated

all_actions = np.array(all_actions)

# ── 按小时聚合 ──────────────────────────────────
hours = np.tile(np.arange(24), len(all_actions) // 24)
hours = hours[:len(all_actions)]

hourly_actions = {}
for h in range(24):
    mask = hours == h
    hourly_actions[h] = all_actions[mask]

fig = go.Figure()

hour_means = [float(np.mean(hourly_actions[h])) for h in range(24)]
hour_stds = [float(np.std(hourly_actions[h])) for h in range(24)]

fig.add_trace(go.Bar(
    x=list(range(24)),
    y=hour_means,
    error_y=dict(type='data', array=hour_stds, visible=True),
    marker=dict(
        color=hour_means,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='归一化投标量'),
    ),
    name='平均投标量',
))

fig.update_layout(
    title=dict(text='PPO 策略各小时平均投标量 (10 episodes)', font=dict(size=16)),
    xaxis=dict(title='Hour of Day', tickmode='array', tickvals=list(range(24))),
    yaxis=dict(title='归一化投标量 [0,1]', range=[0, 1]),
    height=450,
    hovermode='x',
    showlegend=False,
)
fig.show()

print("各小时平均投标量:")
for h in range(24):
    print(f"  Hour {h:2d}: {hour_means[h]:.3f} ± {hour_stds[h]:.3f}")

### 📖 分析投标模式

- **高峰时段 (8-18h)**: 负荷高，投标量通常较高
- **凌晨时段 (0-6h)**: 负荷低，投标量应减少
- **标准差**: 反映策略对不同场景的自适应能力

> 如果 PPO 策略在所有小时都输出接近相同的投标量，
> 说明策略没有学到时间依赖关系——可能需要增加训练步数或调整网络结构。

---
## 思考题

1. PPO 的 reward 比随机策略高多少？为什么 RL 策略在电力交易中可能优于随机策略？

2. 观察训练曲线：Episode reward 有稳定上升的趋势吗？如果没有，可能的原因是什么？（学习率、网络大小、reward 函数设计？）

3. 动作分布直方图显示 PPO 是否学会了「低谷少投、高峰多投」的模式？如果不是，应该怎么改进？\n
4. 当前 reward 函数是 `profit_only`。如果换成 `risk_adjusted` 或 `volume_penalty`，策略行为会有什么变化？

5. 本 notebook 只用了 PPO。如果换用 TD3 或 SAC（Notebook 07），你会预期观察到什么差异？

---
## 下一步
- Notebook 07: 多算法对比 (PPO vs TD3 vs SAC) + 回测
- Notebook 08: 模型可解释性 (SHAP)

**学习笔记**
- 我们成功构建了 Gymnasium 电力交易环境
- 实现了随机策略基线
- 训练了 PPO 智能体并评估了其投标行为